# Exercise 15: Power analyses

This  assignment is designed to give you practice with Monte Carlo methods to conduct power analyses via simulation. You won't need to load in any data for this homework. We will, however, be using parts of the homework from last week.

---
## 1. Simulating data (1 point)


Pull your `simulate_data()` function from your last homework and add it below.

As a reminder, this function simulates the relationship between age, word reading experience, and reading comprehension skill.

`c` is reading comprehension, and `x` is word reading experience.

In [38]:
sample_size = 100 # How many children in data set?
age_lo = 80     # minimum age, in months
age_hi = 200    # maximum age, in months
beta_xa = 0.5   # amount by which experience changes for increase of one month in age
beta_x0 = -5    # amount of experience when age = 0 (not interpretable, since minimum age for this data is 80 months)
sd_x = 50       # standard dev of gaussian noise term, epsilon_x
beta_ca = 0.8   # amount that comprehension score improves for every increase of one unit in age
beta_cx = 3     # amount that comprehension score improves for every increase of one unit in reading experience
beta_c0 = 10    # comprehension score when reading experience is 0.
sd_c = 85      # standard dev of gaussian noise term, epsilon_c

simulate_data <- function(sample_size, age_lo, age_hi, beta_xa, beta_x0, sd_x, beta_ca, beta_cx, beta_c0, sd_c) {
        noise_c = rnorm(sample_size, mean = 0, sd = sd_c) #reading comprehension noise term
        noise_x = rnorm(sample_size, mean = 0, sd = sd_x) #word reading exp noise term

        age = runif(sample_size, min = age_lo, max = age_hi) #generate n random values from specified range

        x = beta_xa*age + beta_x0 + noise_x #reading experience
        c = beta_ca*age + beta_cx*x + beta_c0 + noise_c #reading comprehension
                
      return(data.frame(age=age,x=x,c=c)) # it's actually bad form to have a variable named "c" in R, my bad...
}

dat <- simulate_data(sample_size, age_lo, age_hi, beta_xa, beta_x0, sd_x, beta_ca, beta_cx, beta_c0, sd_c)
head(dat)

,age,x,c
,<dbl>,<dbl>,<dbl>
1,96.2013,5.881996,95.39665
2,91.0133,-46.551451,-149.55793
3,122.5741,82.212108,409.03197
4,116.3957,91.556814,391.90983
5,153.9042,145.196087,473.12525
6,198.9908,50.878117,272.17445


---
## 2. `run_analysis()` function (2 points)

Last week, we looked at whether word reading experience(`x`) mediated the relation between `age` and reading comprehension (`c`).

Now we're going to use our `simulate_data()` function to conduct a power analysis. The goal is to determine how many participants we would need in order to detect both the mediated and the direct effects in this data.

*Note: We're going to pretend for the sake of simplicity that we don't have any control over the ages of the children we get (so ages are generated using `runif(sample_size, age_lo, age_hi)`, although of course this would be an unusual situation in reality.*

First, write a function, `run_analysis()`, that takes in simulated data, runs **your mediation from last week**, and returns a vector containing the ACME and ADE estimates and p-values (these are the `d0`, `d0.p`, `z0`, and `z0.p` features of the mediated model object, e.g., `fitMed$d0.p`). Print this function's output for the data we simulated previously.

In [73]:
library(mediation)
run_analysis <- function(data) {
    fitM <- lm(x ~ age, data=data)
    fitY <- lm(c ~ age + x, data=data)
    fitMed <-mediate(fitM, fitY, treat="age", mediator="x") #run mediation on input data
    #print(summary(fitMed))
    return(c(fitMed$d0, fitMed$d0.p, fitMed$z0, fitMed$z0.p)) #return ACME, ACME p-val, ADE, ADE p-val
}

print(run_analysis(dat))

[1] 2.4160442 0.0000000 0.7667615 0.0160000


---
## 3. `repeat_analysis()` function (3 points)

Next fill in the function `repeat_analysis()` below so that it simulates and analyzes data `num_simulations` times. Store the outputs from each simulation in the `simouts` matrix. Calculate and return the coverage across all the simulations run for both ACME and ADE.

In [66]:
#Tamar: this cell is just to check indexing syntax etc. See cell below for answers
# num_simulations=10
# simouts <- matrix(rep(NA, num_simulations*4), nrow=num_simulations, ncol=4)
# #print(simouts)

# simouts[2,] = run_analysis(dat)
# simouts[3,] = run_analysis(dat)
# simouts[4,] = run_analysis(dat)

# # simouts[1,2] <- 1
# print(simouts)
# #rm(simouts)

# count_nonzero <- sum(simouts[2:4,2] < 0.05)
# count_nonzero

          [,1] [,2]      [,3]  [,4]
 [1,]       NA   NA        NA    NA
 [2,] 2.379166    0 0.7661183 0.012
 [3,] 2.396411    0 0.7826185 0.016
 [4,] 2.395331    0 0.7692264 0.010
 [5,]       NA   NA        NA    NA
 [6,]       NA   NA        NA    NA
 [7,]       NA   NA        NA    NA
 [8,]       NA   NA        NA    NA
 [9,]       NA   NA        NA    NA
[10,]       NA   NA        NA    NA


[1] 3

In [103]:
repeat_analysis <- function(num_simulations, alpha, sample_size, age_lo, age_hi,
        beta_xa, beta_x0, sd_x, beta_ca, beta_cx, beta_c0, sd_c) {
    # Initialize simouts matrix for storing each output from run_analysis()
    simouts <- matrix(rep(NA, num_simulations*4), nrow=num_simulations, ncol=4)
    # Start simulating
    for (i in 1:num_simulations) {
        dat <- simulate_data(sample_size, age_lo, age_hi, beta_xa, beta_x0, sd_x, beta_ca, beta_cx, beta_c0, sd_c)
        simouts[i, ] = run_analysis(dat) }

    # Calculate coverage for both ACME and ADE estimates using p-values in simouts
    ACME_cov = mean(simouts[,2] <= alpha)
    ADE_cov =  mean(simouts[,4] <= alpha)
    
    return(list(ACME_cov = ACME_cov, ADE_cov = ADE_cov))
}

# check <- repeat_analysis(10, 0.05, sample_size, age_lo, age_hi, beta_xa, beta_x0, sd_x, beta_ca, beta_cx, beta_c0, sd_c)
# head(check,10)

Now run the `repeat_analysis()` function using the same parameter settings as above, for 10 simulations, with an alpha criterion of 0.01.

In [72]:
num_simulations = 10
alpha = 0.01
ten_sims <- repeat_analysis(num_simulations, alpha, sample_size, age_lo, age_hi, beta_xa, beta_x0, sd_x, beta_ca, beta_cx, beta_c0, sd_c)

[1] 10  4


---
## 4. Testing different sample sizes (2 points)

Finally, do the same thing (10 simulations, alpha criterion of 0.01) but for 5 different sample sizes: 50, 75, 100, 125, 150. You can do this using `map` (as in the tutorial), or a simple `for` loop, or by calculating each individually. Up to you! This should take around 3 minutes to run.

In [104]:
sample_sizes <- c(50, 75, 100, 125, 150)
num_simulations = 10
alpha = 0.01
cov_by_ss <- matrix(0, nrow = length(sample_sizes), ncol = 2)
# print(cov_by_ss)
# cov_by_ss[,1] <- sample_sizes
# print(cov_by_ss)

for (i in 1:length(sample_sizes)) {
    ten_sims <- repeat_analysis(num_simulations, alpha, sample_sizes[i], age_lo, age_hi, beta_xa, beta_x0, sd_x, beta_ca, beta_cx, beta_c0, sd_c)
    cov_by_ss[i, 1] = ten_sims$ACME_cov #chatGPT helped me with the syntax here
    cov_by_ss[i, 2] = ten_sims$ADE_cov
    rm(ten_sims)
    }

Print your results.

In [105]:
print(cov_by_ss)


     [,1] [,2]
[1,]  0.3  0.3
[2,]  0.8  0.6
[3,]  0.9  0.6
[4,]  0.8  0.8
[5,]  0.9  0.9


## 5. Reflection (2 pts)

If this were a real power analysis, we'd want to run more simulations per sample size (to get a more precise estimate of power) and we may also want to test out some other values of the parameters we used to simulate our data. However, what would you conclude just based on the results above?

> I would conclude that the coverage increases as we increase the sample size, for both the direct and mediated effects. 

Given how we generated the data, why was the direct effect harder to detect than the mediated effect?
> Not sure! We gave c (direct effect) a stronger beta than x (indirect effect), so I'm not sure why the indirect effect is easier to detect. Maybe because x had a smaller gaussian noise term?

**DUE:** 11:59pm EST, March 31, 2026

**IMPORTANT** Did you collaborate with anyone on this assignment? If so, list their names here.
> *Someone's Name*

**GenAI Utilization** Did you utilize any generative AI tools on this assignment? If so, please list the item and the paste respective prompt you used.

>I used chatGPT for debugging/syntax help. For example, I wasn't sure how to add repeat_analysis outputs to the matrix cov_by_ss(in step 4), and it suggested to index into the repeat_analysis output using dollar signs. See specific prompt below:
> * What does this error mean? Error in cov_by_ss[i, 2] <- ten_sims[2]: incorrect number of subscripts on matrix